# Session 8 — FlashAttention

IO-awareness: the bottleneck is memory bandwidth, not FLOPs. Locally (Mac CPU/MPS) we exercise backend *selection*; the real fused CUDA kernel needs a GPU — see `colab/session08_flashattention_colab.ipynb`.

In [1]:
import sys; sys.path.insert(0, '../src')
import torch
from kv_cache_variants.sdpa_backends import available_backends, run_sdpa, default_device
from kv_cache_variants.bench import benchmark

print('device:', default_device(), '| working backends here:', available_backends())

device: mps | working backends here: ['math', 'flash']


In [2]:
q = k = v = torch.randn(2, 8, 512, 64, device=default_device())
for backend in available_backends():
    result = benchmark(run_sdpa, q, k, v, backend=backend, warmup=3, iters=10)
    print(f'{backend}: mean {result["mean_s"]*1000:.3f} ms')

math: mean 1.744 ms
flash: mean 0.535 ms
